# Azure

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/azure/azure.ipynb)

This notebook covers the Azure services a data engineer actually uses, with **Microsoft Fabric** as the default analytics path: OneLake and shortcuts for storage, Lakehouse and Warehouse items, Fabric Data Factory for movement and orchestration, Real-Time Intelligence for streams, and capacity-based billing. Azure Synapse Analytics appears as the existing path many estates still run, with the design rules you need to operate it and the route to migrate off it. The Azure PaaS building blocks that both paths share stay in depth: ADLS Gen2, Azure Data Factory, Azure Databricks with Unity Catalog, Event Hubs, and Entra ID with managed identities. Legacy Windows Azure cloud service material (web and worker roles, cspack, guest OS, swap deployments) is deliberately omitted. The *Things to remember* sections near the end compress the numbers and mistakes worth memorizing.

*Last reviewed: November 2026, against the Azure documentation as of November 2026.*

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.4"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## The Azure mental model for data engineering

Azure's data platform maps onto the same five layers as any cloud lake. In 2026 there are two ways to fill those layers: **Microsoft Fabric**, a SaaS platform where every engine reads and writes one lake (OneLake) and one capacity pays for all of it, and the **Azure PaaS** path, where you assemble ADLS Gen2, Data Factory, Synapse or Databricks, and Event Hubs yourself. New analytics builds should start on Fabric; existing Synapse estates keep running on the PaaS path until they migrate.

| Layer | Fabric (default for new builds) | Azure PaaS (existing estates, fine-grained control) |
|---|---|---|
| Ingest | Data Factory copy jobs and pipelines, mirroring, Eventstream | Azure Data Factory, Event Hubs, Azure Functions |
| Store | OneLake (Delta tables), shortcuts to ADLS, S3, GCS | ADLS Gen2 (Blob Storage with hierarchical namespace) |
| Catalog & govern | OneLake catalog, workspace roles, Microsoft Purview | Unity Catalog (Databricks), Purview, Entra ID + RBAC + POSIX ACLs |
| Process | Spark notebooks, Dataflow Gen2, Warehouse T-SQL | Databricks, Synapse Spark, Data Factory mapping data flows |
| Serve | Warehouse, Lakehouse SQL analytics endpoint, Eventhouse (KQL), Power BI | Synapse dedicated and serverless SQL, Cosmos DB |

Cross-cutting: orchestration (Fabric or Azure Data Factory pipelines, or Airflow), observability (Fabric Capacity Metrics app, Azure Monitor), and cost control (capacity size and pause on Fabric; DWU pause, auto-terminate clusters, and storage tiers on PaaS).

The two paths are not exclusive. A common 2026 shape keeps ADLS Gen2 and Databricks for heavy engineering, and puts a Fabric Lakehouse with shortcuts over the same ADLS paths so analysts query it with T-SQL and Power BI without a copy.

## Microsoft Fabric: OneLake, Lakehouse and Warehouse

**Mental model.** Fabric is one SaaS platform with several engines (Spark, T-SQL, KQL, Power BI) that all store data in **OneLake**, a single logical data lake for the organization. Work is organized into workspaces, and each workspace holds *items*: lakehouses, warehouses, notebooks, pipelines, eventhouses, semantic models. Tables are stored as Delta Lake in OneLake, so a table written by Spark is readable by the SQL engine and by Power BI without an export.

**OneLake and shortcuts.** OneLake exposes a subset of the ADLS Gen2 and Blob APIs, so existing tools address it with a familiar URL:

```text
https://onelake.dfs.fabric.microsoft.com/<workspace>/<lakehouse>.Lakehouse/Tables/orders/
```

A **shortcut** is a OneLake object that points at data somewhere else and behaves like a symbolic link: nothing is copied, deleting the shortcut leaves the target untouched, and moving the target breaks the shortcut. Targets can be other OneLake items (lakehouses, warehouses, KQL databases, mirrored databases) or external storage: ADLS Gen2, Blob Storage, Amazon S3 and S3-compatible stores, Google Cloud Storage, Dataverse, Iceberg tables, and OneDrive or SharePoint. A shortcut under a lakehouse's `Tables` folder that points at a Delta table is registered as a table automatically; shortcuts under `Files` can point at any format. For S3, GCS and gateway shortcuts you can turn on a workspace cache (1 to 28 days) to cut cross-cloud egress.

**Lakehouse versus Warehouse.** Both store Delta tables in OneLake and share the same SQL engine; they differ in how you write.

| | Lakehouse | Warehouse |
|---|---|---|
| Primary authoring | Spark (Python, Scala, Spark SQL, R) | T-SQL |
| Writes | Spark, pipelines, dataflows, shortcuts | `COPY INTO`, `INSERT`, `CREATE TABLE AS SELECT`, pipelines, dataflows |
| SQL access | auto-created **SQL analytics endpoint**: read-only T-SQL (queries, views, table-valued functions, no DML) | full DQL, DML and DDL |
| Transactions | Delta ACID per table | multi-table ACID transactions |
| Data shapes | structured and unstructured (`Files` and `Tables`) | structured only |
| Typical role | bronze and silver zones, data science | gold marts and enterprise warehousing |

The Microsoft decision guide reduces to three questions: Spark or T-SQL, whether you need multi-table transactions, and whether the data includes unstructured files. Many teams use both: a lakehouse for engineering and a warehouse for the served model, joined with cross-database queries that read across items without duplicating data.

```sql
-- In a Fabric Warehouse: three-part names read a lakehouse table through its SQL endpoint, no copy.
CREATE TABLE gold.daily_revenue AS
SELECT CAST(o.order_ts AS DATE) AS order_date, SUM(o.amount) AS revenue
FROM sales_lakehouse.dbo.orders AS o
GROUP BY CAST(o.order_ts AS DATE);
```

**What disappears compared with Synapse.** A Fabric Warehouse has no distribution choice, no index tuning and no DWU slider: the Migration Assistant documentation lists indexes as no longer needed, and the decision guide describes autonomous workload management with nothing to configure. The design levers that remain are file and table layout, statistics, and the size of the capacity.

**Common mistake.** Trying to `INSERT` into a lakehouse through its SQL analytics endpoint. The endpoint is read-only; write with Spark or a pipeline, or use a Warehouse when the team works in T-SQL.

**Gotcha.** Shortcuts delegate access. An internal shortcut authorizes with the caller's identity at the target, but some paths (Direct Lake over SQL, T-SQL in delegated identity mode) pass the item owner's identity instead. Check which mode a semantic model or endpoint uses before assuming row-level permissions at the target still apply.

## Fabric Data Factory and Real-Time Intelligence

**Fabric Data Factory** is described by Microsoft as the next generation of Azure Data Factory. It keeps the pipeline model (activities, control flow, schedules and event triggers) and adds options that need less configuration:

+ **Copy job**: the preferred way to move data, with bulk, incremental, and change data capture delivery from many sources to many destinations, without building a pipeline.
+ **Copy activity in a pipeline**: the classic, fully configurable copy with manual control of parallelism, used when the copy is one step of a larger workflow.
+ **Mirroring**: a near real-time replica of an operational database in OneLake, for analytics without a hand-built CDC pipeline.
+ **Dataflow Gen2**: low-code Power Query transforms that land in a lakehouse or warehouse.
+ **Notebook, Spark job definition, stored procedure and dbt job activities**: run code-first transforms from a pipeline.
+ **Apache Airflow job**: Airflow DAGs hosted inside Fabric, for teams that prefer orchestration as Python.

Linked services become *connections*, datasets are folded into the activity, and there is no integration runtime to size; on-premises sources go through the on-premises data gateway. Microsoft publishes an upgrade guide for moving Azure Data Factory and Synapse pipelines into Fabric.

**Real-Time Intelligence** is Fabric's streaming workload:

+ **Eventstream** connects sources such as Event Hubs, IoT Hub, Kafka, Kinesis, Google Pub/Sub and database CDC feeds, then filters, aggregates over windows, and routes events to destinations without code.
+ **Eventhouse** (KQL databases) stores time-based events indexed by arrival time and answers queries in KQL or T-SQL; data can be made available in OneLake for the other engines.
+ **Real-Time hub** is the catalog of streams the organization can discover and reuse.
+ **Activator** watches streams, KQL results or Power BI visuals and takes an action when a condition holds: alert a person, run a pipeline, notebook or dataflow, or start a Power Automate flow.

**Worked example.** Telemetry already lands in Event Hubs. An Eventstream reads the hub, drops malformed events, writes raw events to an Eventhouse for second-level dashboards, and writes one-minute aggregates to a lakehouse table. An Activator rule runs a maintenance pipeline when a device reports over-temperature three times in ten minutes.

**Common mistake.** Rebuilding every Azure Data Factory pipeline by hand on day one. Mirroring or a copy job often replaces a whole family of CDC and incremental-copy pipelines; migrate the orchestration that still adds value and retire the rest.

## Fabric capacity: how billing and throttling work

**Mental model.** Fabric does not bill per engine. You buy a **capacity**, an F SKU sized in capacity units (CUs), and every workspace assigned to it draws from the same pool: Spark, warehouse queries, pipelines, eventstreams and Power BI all consume CU-seconds. F SKUs start at F2 (2 CUs) and double from there (F4, F8 ... F64 ... F2048 and above). F capacities are bought through Azure and billed per second with a one-minute minimum and no commitment, with an optional yearly reservation for a discount. A pay-as-you-go capacity can be paused, which stops billing.

**Bursting and smoothing.** An operation may use more compute than the SKU provides (bursting), and Fabric then spreads the CU cost over future 30-second *timepoints* (smoothing). Interactive operations are smoothed over at least 5 minutes and up to 64 minutes; background operations, which include most warehouse and pipeline work, are smoothed over 24 hours. That is why a large nightly job does not throttle a small capacity: a 1 CU-hour background job on an F2 adds only about 2.1% to each timepoint.

**Throttling.** Throttling depends on how much *future* capacity the smoothed usage has already consumed:

| Future capacity used | Effect |
|---|---|
| up to 10 minutes | overage protection, no throttling |
| 10 to 60 minutes | new interactive operations are delayed 20 seconds |
| 60 minutes to 24 hours | new interactive operations are rejected; background jobs still start |
| over 24 hours | all new operations are rejected until the overage burns down |

Operations already running are never throttled. Real-Time Intelligence skips the 20-second delay stage, and eventstreams are slowed rather than stopped.

**Licensing edges that affect design.** On F64 and larger, users with a free Fabric license and a viewer role can read Power BI content; below F64, every viewer needs a Pro or Premium Per User license. Premium Per User does not provide a Fabric capacity, so it cannot run lakehouses, warehouses or notebooks.

**Common mistake.** Sizing a capacity from its peak in the first hour of a big load. Because background work is smoothed over 24 hours, size from the 24-hour total of background CU-seconds plus the interactive peak, and watch the Capacity Metrics app for carryforward before scaling up.

**Gotcha.** Pausing a capacity bills the accumulated smoothed usage immediately, and content on the capacity is unavailable while it is paused. Pause dev and test capacities on a schedule; do not pause the capacity behind production reports to clear throttling unless you accept that outage.

In [ ]:
TIMEPOINT_SECONDS = 30
TIMEPOINTS_PER_DAY = 24 * 60 * 60 // TIMEPOINT_SECONDS
BACKGROUND_SMOOTHING_TIMEPOINTS = TIMEPOINTS_PER_DAY


def background_share_per_timepoint(job_cu_seconds, capacity_cus):
    """Share of each 30-second timepoint a background job uses once smoothed over 24 hours."""
    smoothed_cu_seconds = job_cu_seconds / BACKGROUND_SMOOTHING_TIMEPOINTS
    timepoint_capacity = capacity_cus * TIMEPOINT_SECONDS
    return smoothed_cu_seconds / timepoint_capacity


def throttle_stage(future_minutes_used):
    """Fabric throttling stage from the minutes of future capacity already consumed."""
    if future_minutes_used <= 10:
        return "overage protection"
    if future_minutes_used <= 60:
        return "interactive delay"
    if future_minutes_used <= 24 * 60:
        return "interactive rejection"
    return "background rejection"


ONE_CU_HOUR = 3600
share_f2 = background_share_per_timepoint(ONE_CU_HOUR, capacity_cus=2)
share_f64 = background_share_per_timepoint(ONE_CU_HOUR, capacity_cus=64)
print(f"timepoints per day = {TIMEPOINTS_PER_DAY}")
print(f"1 CU-hour background job on F2  -> {share_f2:.2%} of each timepoint")
print(f"1 CU-hour background job on F64 -> {share_f64:.3%} of each timepoint")
for minutes in (5, 30, 240, 2000):
    print(f"{minutes:>5} future minutes used -> {throttle_stage(minutes)}")

assert TIMEPOINTS_PER_DAY == 2880
assert round(share_f2 * 100, 1) == 2.1
assert share_f64 < share_f2
assert throttle_stage(10) == "overage protection"
assert throttle_stage(45) == "interactive delay"
assert throttle_stage(600) == "interactive rejection"
assert throttle_stage(1441) == "background rejection"

## Designing an Azure data lake

On the PaaS path, use ADLS Gen2 (hierarchical namespace enabled) for storage, Azure Data Factory for orchestration, Databricks for processing, Unity Catalog or Purview for governance, and Power BI for serving. On Fabric, the same zones become lakehouses (often one per zone) in OneLake, and shortcuts let a Fabric lakehouse sit over existing ADLS paths without copying them. Either way, organize storage into raw, curated, and serving layers.

```text
abfss://lake@acme.dfs.core.windows.net/raw/orders/ds=2026-10-04/
abfss://lake@acme.dfs.core.windows.net/curated/orders/ds=2026-10-04/
abfss://lake@acme.dfs.core.windows.net/marts/fact_orders/
```

Good answers include identity and security: managed identities instead of keys, RBAC plus POSIX ACLs on ADLS paths, private endpoints, Key Vault for secrets, and diagnostic logs to Log Analytics.

**Mental model.** A lake is organized by trust and refinement, not by source system. The usual zones are raw (landing, source-shaped), curated or silver (typed, deduplicated, conformed), and serving or gold (aggregated, modeled for a specific consumer). ADLS Gen2 with hierarchical namespace gives real directories and atomic directory renames, which object-only blob storage does not have.

**How it works.** Files land as Parquet or Delta, partitioned on the column that filters most often, usually a date. Managed identities read and write; POSIX ACLs on directories plus RBAC roles control who sees which zone. Lifecycle rules move cold partitions from Hot to Cool and then Archive, and Purview or Unity Catalog records schemas, owners, and lineage.

**Worked example.** Raw data sits at `abfss://lake@acmedata.dfs.core.windows.net/raw/orders/ds=2026-10-04/`, curated data is a Delta table at `curated/orders/`, and the serving layer is a Fabric Warehouse table (or, on an existing estate, a Synapse dedicated table) built by `CTAS`. A lifecycle rule moves `raw/orders/` to Cool after 30 days and deletes it after 365.

**Common mistake.** Building one flat container with no zones, or leaving every zone in the Hot tier forever. Both make the lake expensive and hard to govern. Another common slip is using the `blob` endpoint for a lake; directories there are only name prefixes, so renames and ACLs behave differently.

**Gotcha.** A lake is not a dumping ground. Without a catalog and a retention policy, raw, curated, and personal copies of the same table accumulate until nobody can tell which one is correct.

## Choosing the compute: Fabric, Data Factory, Databricks, Synapse, or Functions

Start new analytics work on Fabric: Lakehouse plus Spark notebooks for engineering, Warehouse for T-SQL marts, Data Factory copy jobs, pipelines or mirroring for movement, and Real-Time Intelligence for streams. Use Azure Databricks when you need the deepest Spark and Delta tuning, ML tooling, multi-cloud portability, or already standardize on Unity Catalog. Keep Azure Data Factory for estates that are not on Fabric. Keep Synapse for existing dedicated pools until they migrate; do not start new Synapse workspaces for greenfield work. Use Azure Functions for small event-driven automation.

Example choices:

+ nightly copy from SQL Server into the lake: Fabric copy job or mirroring (Azure Data Factory on a PaaS estate)
+ large Spark transformation over Delta tables: Fabric Spark or Databricks
+ SQL serving layer for Power BI: Fabric Warehouse (Synapse dedicated pool on an existing estate)
+ validate a file when it lands in blob storage: Azure Functions

**Mental model.** Pick the engine by the shape of the work and by which billing model you want. Fabric bills one shared capacity for every engine. Data Factory is an orchestrator and connector library: it moves data and schedules work, and it can run visual transforms but is not built for heavy joins. Databricks is a Spark and Delta platform tuned for large transformations and machine learning, billed in DBUs. Synapse combines provisioned SQL pools, serverless SQL and Spark in one workspace. Functions is small, event-driven glue.

**How it works.** Azure Data Factory copy activities run on an integration runtime, so throughput is a configuration question (DIUs, parallelism, staging). Synapse dedicated pools store data in 60 distributions behind a clustered columnstore index and are sized in DWUs. Databricks separates driver and workers and bills DBUs per cluster-hour. Fabric engines draw CU-seconds from the capacity the workspace is assigned to. Functions scales per event and has a time limit, so it suits webhooks and small files, not shuffles.

**Worked example.** Ingest 400 source tables with a Fabric copy job (breadth of connectors, incremental and CDC modes), clean and join them in Spark (Fabric notebooks or Databricks), publish marts to a Fabric Warehouse for Power BI, and use a Function to validate a webhook payload.

**Common mistake.** Forcing one service to do everything: putting a large join in a low-code dataflow, or running a long shuffle in Functions. Match the tool to the workload and let orchestration stay orchestration.

## Blob Storage: the object store under ADLS

Blob Storage is Azure's object store: an account holds containers, a container holds blobs, and a blob is a byte sequence addressed by a URL. The URL format is `https://<account>.blob.core.windows.net/<container>/<blob>`, or `https://<account>.dfs.core.windows.net/<container>/<path>` for the ADLS Gen2 (DFS) endpoint. It is the cheapest durable layer and the foundation of ADLS Gen2, and OneLake exposes a compatible subset of the same APIs.

**How it works.** Data files are block blobs: a client stages blocks and commits a block list, so a large upload can resume and commit atomically. Append blobs suit log tails. Durability comes from replication: locally redundant within a datacenter, zone redundant across zones, or geo redundant across regions. Access tiers (Hot, Cool, Cold, Archive) trade storage cost against access cost, and lifecycle rules move data between them.

**Worked example.** A pipeline writes Parquet files to `raw/orders/ds=2026-10-04/`, and a lifecycle rule moves that prefix to Cool after 30 days and deletes it after 365.

**Common mistake.** Using blob storage as a database. Listing millions of small files is slow and expensive, and there is no transactional update. Compact files, partition paths, and keep table state in Delta or Iceberg metadata and a catalog.

## Queue Storage, Service Bus, and dead-letter queues

Azure Queue Storage is a simple REST-based get/put/peek queue for reliable persistent messaging within and between services. Service Bus Queues are part of a richer messaging infrastructure that supports queuing plus publish/subscribe, sessions, transactions, dead-lettering, duplicate detection, and integration patterns. Use Queue Storage for cheap high-volume decoupling and Service Bus for ordered, transactional, or topic-based messaging.

**Mental model.** Queue Storage is a simple, cheap, at-least-once queue with messages up to 64 KB and a maximum time to live of seven days. Service Bus is a messaging broker with ordering sessions, transactions, duplicate detection, dead-lettering, larger messages, and richer delivery semantics.

**How it works.** Queue Storage gives visibility timeouts and a poison-message count but no native ordering and no sessions. Service Bus adds FIFO with sessions, publish-subscribe through topics and subscriptions, scheduled messages, and dead-letter queues with a reason. Standard Service Bus messages are up to 256 KB; Premium goes to 100 MB.

**Worked example.** Use Queue Storage for a cheap image-thumbnail backlog; use Service Bus when an order must be processed in sequence per customer and a failed message must be parked with a reason.

**Common mistake.** Assuming Queue Storage preserves order or handles poisonous messages for you. After the dequeue count is exceeded you must move the message yourself, and ordering is not guaranteed.

A dead-letter queue holds messages the system could not process. Messages are moved there when:

+ A message expires and dead-lettering for expired messages is enabled.
+ The max delivery count is exceeded on a queue or subscription.
+ A filter evaluation exception occurs on a subscription and dead-lettering on filter exceptions is enabled.

**Mental model.** A dead-letter queue (DLQ) is where a broker parks messages it cannot deliver or that a consumer explicitly rejects. It is a safety net, not a trash can.

**How it works.** Service Bus moves messages to its DLQ on expiration, on exceeding the delivery count, on a filter evaluation error, or when the consumer calls dead-letter explicitly. Queue Storage has no DLQ, so you must implement one. The DLQ keeps the message body and the reason, so a message can be inspected and replayed after a fix.

**Worked example.** An order message fails five times because of a bad currency code. Service Bus dead-letters it with reason `MaxDeliveryCountExceeded`; an operator fixes the mapping, then a replay tool resubmits from the DLQ.

**Common mistake.** Letting the DLQ grow unwatched. Alert on DLQ depth, record the reason code, and build a replay path; otherwise failures are silent.

## Synapse SQL on existing estates

Synapse is the path many Azure estates already run, so you still need to operate it well. For new warehouses, prefer a Fabric Warehouse (no distributions or indexes to design); the rules below apply to the Synapse pools you inherit and to planning their migration.

Synapse SQL has a serverless pool for querying files in place and dedicated pools for provisioned warehouse performance. Dedicated pools use distributions (hash, round-robin, replicated) and a clustered columnstore index; loads use `COPY INTO` from ADLS rather than row inserts.

```sql
-- Serverless: query Parquet directly from ADLS Gen2
SELECT TOP 10 *
FROM OPENROWSET(
  BULK 'https://acmedata.dfs.core.windows.net/lake/curated/orders/**',
  FORMAT = 'PARQUET'
) AS orders;

-- Dedicated pool: modeled table with hash distribution and clustered columnstore
CREATE TABLE dbo.fact_orders (
  order_id    BIGINT NOT NULL,
  customer_id BIGINT NOT NULL,
  order_ts    DATETIME2 NOT NULL,
  amount      DECIMAL(12,2) NOT NULL
)
WITH (DISTRIBUTION = HASH(customer_id), CLUSTERED COLUMNSTORE INDEX);

COPY INTO dbo.fact_orders
FROM 'https://acmedata.dfs.core.windows.net/lake/curated/orders/'
WITH (FILE_TYPE = 'PARQUET', CREDENTIAL = (IDENTITY = 'Managed Identity'));
```

**Mental model.** Synapse SQL has two engines. Serverless SQL queries files in ADLS in place and bills per terabyte scanned. Dedicated SQL pool is a provisioned MPP warehouse: every table is spread over 60 distributions, and you pay for DWUs whether you query or not. Choose dedicated for repeated, high-concurrency BI; choose serverless for exploration and occasional queries over the lake.

**How it works.** In a dedicated pool, a table's distribution and index decide how much data moves at query time. Hash distribution colocates rows with the same key so joins on that key avoid a shuffle. Replicated tables copy a small dimension to every distribution and remove the shuffle entirely. Round-robin spreads load evenly when there is no good join key, but every join then needs movement. The default index is a clustered columnstore, which compresses well and is fast for scans; heaps and clustered indexes are for staging and point lookups. Statistics tell the optimizer how many rows to expect, and a stale statistic produces a bad plan.

**Worked example.** A fact table joins a large customer dimension and a small date dimension on every query. Hash-distribute the fact on `customer_id`, replicate the date dimension, and keep the customer dimension hash-distributed on `customer_id` as well.

```sql
CREATE TABLE dbo.dim_date
WITH (DISTRIBUTION = REPLICATE, CLUSTERED COLUMNSTORE INDEX)
AS SELECT * FROM staging.dim_date;

CREATE TABLE dbo.fact_sales
WITH (DISTRIBUTION = HASH(customer_id), CLUSTERED COLUMNSTORE INDEX)
AS SELECT * FROM staging.fact_sales;

CREATE STATISTICS st_fact_sales_region ON dbo.fact_sales (region);
```

**Common mistake.** Distributing every table round-robin and then joining large tables. Each join adds a data movement step, and the plan fills with ShuffleMove operations that dominate runtime. A second common mistake is a clustered columnstore index on a table with only a few thousand rows: it lands in the delta store and performs worse than a heap.

## Building an Azure Data Factory pipeline

Azure Data Factory remains the orchestrator on estates that are not on Fabric, and its JSON model carries over: Fabric Data Factory pipelines use the same activity and control-flow concepts, with connections in place of linked services and no integration runtime to size.

Data Factory pipelines are JSON activities driven by datasets and linked services. A copy activity reads a source dataset and writes a sink dataset, and triggers (schedule, tumbling window, or event) start the pipeline. Use parameterized datasets and a `ForEach` over a date range for backfills.

```json
{
  "name": "PL_Orders_Daily",
  "properties": {
    "parameters": {"windowStart": {"type": "String"}, "windowEnd": {"type": "String"}},
    "activities": [
      {
        "name": "CopyOrdersToCurated",
        "type": "Copy",
        "typeProperties": {
          "source": {
            "type": "ParquetSource",
            "storeSettings": {"type": "AzureBlobFSReadSettings", "recursive": true}
          },
          "sink": {
            "type": "ParquetSink",
            "storeSettings": {"type": "AzureBlobFSWriteSettings"}
          },
          "enableStaging": false
        },
        "inputs": [{"referenceName": "DS_RawOrders", "type": "DatasetReference",
          "parameters": {"folderPath": "@concat('raw/orders/ds=', pipeline().parameters.windowStart)"}}],
        "outputs": [{"referenceName": "DS_CuratedOrders", "type": "DatasetReference",
          "parameters": {"folderPath": "@concat('curated/orders/ds=', pipeline().parameters.windowStart)"}}]
      }
    ]
  }
}
```

Interview points: use managed identity for ADLS access, store secrets in Key Vault rather than linked-service credentials, prefer a tumbling-window trigger for exactly-once time slices, and use the data factory's mapping data flows only when a Spark-free visual transform is genuinely simpler than Databricks.

**Mental model.** Data Factory is a control plane: linked services describe connections, datasets describe data shapes, activities do work, and pipelines compose activities. An integration runtime is the compute that actually moves bytes. Azure IR is managed and serverless, a self-hosted IR runs on your own machines for on-premises and private networks, and Azure-SSIS IR runs SSIS packages.

**How it works.** A copy activity reads a source and writes a sink. Throughput is set by data integration units (DIUs), copy parallelism, and whether staging is enabled. Mapping data flows run on a Spark cluster and are billed by vCore-hour, so they suit visual transforms, not heavy joins. Triggers start pipelines: schedule, tumbling window (exactly-once time slices with state), and event-based (for example, blob created).

**Worked example.** Copy 2 TB of Parquet from an on-premises file share to ADLS in a four-hour window. Size the copy by DIUs and parallelism, and enable staging when the sink is a dedicated SQL pool.

```json
{
  "name": "CopyOrdersToCurated",
  "type": "Copy",
  "typeProperties": {
    "source": { "type": "ParquetSource" },
    "sink": { "type": "ParquetSink" },
    "dataIntegrationUnits": 32,
    "parallelCopies": 8,
    "enableStaging": true,
    "stagingSettings": {
      "linkedServiceName": { "referenceName": "LS_StagingLake", "type": "LinkedServiceReference" },
      "path": "staging/orders"
    }
  }
}
```

**Common mistake.** Raising DIUs while `parallelCopies` stays at one, or leaving a single self-hosted IR node for a very large copy. Both leave the bottleneck on the source reader. Also do not put a large join in a mapping data flow when Databricks is already in the stack; it duplicates cost and tooling.

## ADLS Gen2 paths and access control

ADLS Gen2 is Blob Storage with a hierarchical namespace, so paths behave like a file system and support POSIX-like ACLs at directory and file level, plus Azure RBAC at the account/container level. Use `abfss://` in Spark and `https://<account>.dfs.core.windows.net` in SQL and the CLI.

```text
abfss://lake@acmedata.dfs.core.windows.net/raw/orders/ds=2026-10-04/
abfss://lake@acmedata.dfs.core.windows.net/curated/orders/ds=2026-10-04/
```

```bash
az storage fs access set --account-name acmedata --file-system lake --path raw \
  --acl "user:11111111-2222-3333-4444-555555555555:r-x" --auth-mode login
az storage fs access show --account-name acmedata --file-system lake --path raw
```

**Mental model.** ADLS Gen2 is Blob Storage with hierarchical namespace enabled. Directories become real objects, so a directory rename is one atomic metadata operation instead of a copy of every blob. Access is controlled by two independent systems that are both evaluated: Azure RBAC at account, container, or path scope, and POSIX-style ACLs on directories and files.

**How it works.** A managed identity or user needs a data-plane role such as Storage Blob Data Contributor and, for restricted paths, an ACL entry that allows the operation. ACL evaluation walks from the container root down to the object; a missing execute permission on an intermediate directory blocks access even if the file has an allow entry. The mask on a directory limits what named users and groups can grant themselves.

**Worked example.** Grant a Synapse workspace identity write access to one zone only, without touching the rest of the lake.

```bash
az storage fs access set --account-name acmedata --file-system lake --path curated \
  --acl "user:<workspace-principal-id>:rwx" --auth-mode login

az role assignment create --assignee-object-id <workspace-principal-id> \
  --assignee-principal-type ServicePrincipal \
  --role "Storage Blob Data Contributor" \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-data/providers/Microsoft.Storage/storageAccounts/acmedata
```

**Common mistake.** Granting the role at account scope and believing that secures the lake, or granting an ACL on a file without execute on its parent directories. Also remember that the `blob` endpoint does not expose directories, so ACLs only apply on the `dfs` endpoint.

**Gotcha.** Lifecycle management and ACLs are separate. A lifecycle rule can delete data that the team that owns it still expects to exist, so scope rules by prefix and test them on a small path first.

## Writing a Databricks notebook and PySpark job

In Databricks, notebooks are the interactive surface and `dbutils` provides workspace utilities. Jobs run the same code on a cluster; secrets come from secret scopes, and tables are usually Delta Lake managed by Unity Catalog. Prefer `MERGE` for idempotent upserts and `OPTIMIZE` for compaction.

```python
# Databricks notebook: read raw JSON from ADLS, type and dedupe, MERGE into a Delta table, then OPTIMIZE.
from pyspark.sql import functions as F

raw_path = "abfss://lake@acmedata.dfs.core.windows.net/raw/orders/"
orders = (
    spark.read.json(raw_path)
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("amount", F.col("amount").cast("decimal(12,2)"))
    .dropDuplicates(["order_id"])
)
orders.write.format("delta").mode("overwrite").saveAsTable("curated.orders_bronze")

spark.sql("""
merge into curated.orders as target
using curated.orders_bronze as source
on target.order_id = source.order_id
when matched then update set target.amount = source.amount, target.order_ts = source.order_ts
when not matched then insert *
""")
spark.sql("optimize curated.orders zorder by (customer_id)")
```

The same upsert logic runs locally in DuckDB, which supports `MERGE INTO` with the same semantics:

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE TABLE orders AS
    SELECT * FROM (VALUES (1, 10.00, TIMESTAMP '2026-10-03 09:00'), (2, 20.00, TIMESTAMP '2026-10-03 10:00'))
        t(order_id, amount, order_ts)
""")
# A raw batch with a duplicate redelivery, an update to order 2, and a new order 3.
con.execute("""
    CREATE TABLE raw_batch AS
    SELECT * FROM (VALUES
        (2, '25.00', '2026-10-04 08:00:00'), (2, '25.00', '2026-10-04 08:00:00'), (3, '30.00', '2026-10-04 09:30:00')
    ) t(order_id, amount, order_ts)
""")
con.execute("""
    CREATE TABLE bronze AS
    SELECT DISTINCT order_id, CAST(amount AS DECIMAL(12, 2)) AS amount, CAST(order_ts AS TIMESTAMP) AS order_ts
    FROM raw_batch
""")
con.execute("""
    MERGE INTO orders AS target
    USING bronze AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN UPDATE SET amount = source.amount, order_ts = source.order_ts
    WHEN NOT MATCHED THEN INSERT (order_id, amount, order_ts) VALUES (source.order_id, source.amount, source.order_ts)
""")
result = con.execute("SELECT order_id, CAST(amount AS DOUBLE) FROM orders ORDER BY order_id").fetchall()
print(result)
assert result == [(1, 10.0), (2, 25.0), (3, 30.0)]

Interview points: Unity Catalog gives three-level names (`catalog.schema.table`) and centralized governance; Delta gives ACID and time travel; `zorder` speeds filtered reads on the column you filter most; Delta Live Tables or Lakeflow pipelines add declarative expectations and lineage.

**Mental model.** A Databricks notebook is the interactive surface; a job runs the same code on a schedule against a cluster. Clusters separate a driver from workers, and Delta Lake adds ACID transactions, schema enforcement, and time travel on top of Parquet in ADLS.

**How it works.** Jobs can autoscale workers between a minimum and maximum and can use job clusters that terminate when the run finishes. Unity Catalog gives three-level names (`catalog.schema.table`), centralized grants, and lineage. Delta maintenance keeps files healthy: `OPTIMIZE` compacts small files, and `ZORDER` clusters data on the columns you filter most. Photon accelerates SQL and DataFrame workloads at extra DBU cost.

**Worked example.** An ingest job reads JSON from `raw/`, types and deduplicates it, merges into a bronze Delta table on a business key, and then compacts the table.

**Common mistake.** Reading and writing many tiny files, or partitioning a small table by a high-cardinality column. Both create an explosion of files that slows every later job. Aim for files in the hundreds of megabytes and partition only on columns that are genuinely selectable and low enough in cardinality.

**Gotcha.** `mode("overwrite")` on a whole table is cheap to write and expensive to read later if it leaves thousands of small files behind. Follow bulk writes with compaction.

## Entra ID RBAC and managed identities

Microsoft Entra ID (formerly Azure AD) is the identity plane. Assign Azure roles at the right scope (management group, subscription, resource group, or resource) and use managed identities so services authenticate without stored secrets. For data access, the common roles are `Storage Blob Data Reader/Contributor`, `Synapse SQL Administrator`, and Databricks workspace/Unity Catalog grants.

```bash
# Grant a managed identity read access to the lake account at resource scope
az role assignment create --assignee-object-id <principal-id> --assignee-principal-type ServicePrincipal \
  --role "Storage Blob Data Contributor" \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-data/providers/Microsoft.Storage/storageAccounts/acmedata

# Let a Synapse workspace use its managed identity to load data
az synapse workspace show --name syn-acme --resource-group rg-data --query identity.principalId
az storage fs access set --account-name acmedata --file-system lake --path curated \
  --acl "user:<principal-id>:rwx" --auth-mode login
```

**Mental model.** Microsoft Entra ID is the identity plane; Azure RBAC is the authorization plane. A principal (user, group, service principal, or managed identity) gets a role assignment at a scope, and permissions inherit down the scope tree from management group to resource. Managed identities give a service an Entra ID identity with no stored secret.

**How it works.** A system-assigned identity lives and dies with the resource; a user-assigned identity is a separate resource that several services can share. Workload identity federation lets a CI/CD pipeline exchange an external token for an Azure token without a client secret. Deny assignments override allows, and PIM makes privileged roles eligible rather than permanent.

**Worked example.** Give a Data Factory managed identity read access to one container, and confirm the effective assignment.

```bash
az datafactory show --name adf-acme --resource-group rg-data --query identity.principalId -o tsv

az role assignment create --assignee-object-id <principal-id> \
  --assignee-principal-type ServicePrincipal --role "Storage Blob Data Reader" \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-data/providers/Microsoft.Storage/storageAccounts/acmedata/blobServices/default/containers/lake

az role assignment list --assignee <principal-id> --all -o table
```

**Common mistake.** Assigning roles to individual users instead of groups, and assigning at subscription scope when one container would do. Both make least privilege impossible to audit. Also remember that control-plane roles such as Contributor do not grant data-plane access; you still need a Storage Blob Data role and, for ADLS, the path ACLs.

**Gotcha.** A managed identity is not a security boundary on its own. Whoever can deploy to that resource can use its identity, so pair identities with scoped roles and with deployment controls.

## Streaming events with Azure Event Hubs

Event Hubs is the Azure streaming ingestion service: a partitioned event log with consumer groups, retention (1-90 days depending on tier), and capture to ADLS or Blob Storage. Producers send events with a partition key; consumers read with an offset using the EventProcessorClient or Spark Structured Streaming.

```python
from azure.eventhub import EventData, EventHubProducerClient

producer = EventHubProducerClient.from_connection_string(conn_str=connection_str, eventhub_name="orders")
with producer:
    batch = producer.create_batch(partition_key="customer-42")  # same key -> same partition -> ordered
    for order in orders:
        batch.add(EventData(json.dumps(order)))
    producer.send_batch(batch)
```

**Mental model.** Event Hubs is a partitioned, replayable event log, not a queue. Ordering is guaranteed within a partition only. Throughput units (Standard) or processing units (Premium) buy ingress and egress capacity, and the partition count sets the maximum consumer parallelism per consumer group.

**How it works.** A producer sends events with an optional partition key; the same key always lands in the same partition, which preserves per-key order. Consumers read from an offset in a consumer group, so several independent readers can each see the full stream. The recommended consumer uses the EventProcessorClient with a blob checkpoint store: it owns partitions through leases and records progress. Event Hubs Capture writes Avro micro-batches to ADLS or Blob Storage on a time and size window, so you get a durable log without writing a consumer. On Fabric, an Eventstream can read the same hub and route events to an Eventhouse or a lakehouse without a custom consumer.

## Choosing between Azure data services: trade-offs

+ Microsoft Fabric - default for new analytics: one lake, one capacity, Spark, T-SQL, KQL and Power BI together; weaker when you need fine-grained infrastructure control or per-engine billing.
+ Fabric Lakehouse - Spark-first storage of structured and unstructured data; its SQL endpoint is read-only.
+ Fabric Warehouse - T-SQL warehouse with multi-table transactions and nothing to tune physically; not yet full T-SQL parity with SQL Server or Synapse (check the surface-area page before migrating).
+ ADLS Gen2 - durable lake storage with file semantics; not a database, no transactions or row-level updates.
+ Azure Data Factory - orchestrator and connector-heavy copy tool for estates not on Fabric; not a compute engine for heavy transforms.
+ Azure Databricks - best for large Spark/Delta workloads, ML, and advanced tuning; heavier cost model and cluster management.
+ Synapse - existing path: unified SQL plus Spark plus pipelines; keep running inherited pools, but plan new builds on Fabric.
+ Synapse SQL serverless - pay-per-query over files; poor for high-concurrency repeated BI.
+ Synapse SQL dedicated - high-concurrency warehouse you must design (distributions, indexes, DWUs); not for tiny point lookups or OLTP.
+ Azure Functions - small event-driven automation; avoid long-running or memory-heavy transforms.
+ Event Hubs - high-throughput streaming ingestion; not a task queue (use Service Bus or Storage Queues) and not a long-term store beyond retention.
+ Cosmos DB - globally distributed low-latency NoSQL; not for analytical scans or ad-hoc joins.

Cost drivers: capacity SKU, pause schedule and smoothing headroom (Fabric), cluster DBUs and VM hours (Databricks), DWU/vCore hours and pause behavior (Synapse SQL), activity runs and data movement units (Data Factory), throughput units and retention (Event Hubs), storage transactions and tiering (ADLS), and provisioned RU/s (Cosmos DB). Security drivers: Entra ID RBAC, managed identities, workspace roles and item permissions, POSIX ACLs, private endpoints, Key Vault, and Purview classification. Performance drivers: file size and Parquet/Delta layout, partitioning, distribution keys on Synapse, shuffle, and cluster or capacity sizing.

## Designing a Synapse dedicated SQL pool for performance

This is the existing-estate skill set: tuning a pool you inherit, and understanding what its design encodes before you migrate it. A Fabric Warehouse makes most of these choices for you.

**Mental model.** A dedicated SQL pool spreads every table across 60 distributions, and each query is planned as parallel work over those distributions. Performance is decided by what the plan has to move between them. If a join's keys are already colocated, the join runs locally on each distribution. If not, the engine inserts a data movement step, called a shuffle or broadcast, and that step often dominates runtime.

**How it works.** There are three distribution choices. `HASH` colocates rows by key, so joins and aggregations on that key stay local. `REPLICATE` copies the whole table to every distribution, which removes movement for a small dimension at the cost of storage and load time. `ROUND_ROBIN` spreads rows evenly with no key, which is fine for staging but forces movement on every join.

The index choice follows the workload. A clustered columnstore index compresses columnar data heavily and scans fast, and it is the default for fact tables. It pays off once a table is large enough to fill rowgroups, on the order of a million rows; below that, rows sit in the delta store and a heap or clustered index performs better. Heaps suit staging tables that are loaded and immediately read once. A clustered index suits point lookups and small tables.

Statistics are the optimizer's estimate of row counts, and a stale estimate produces a bad plan. Auto-create statistics is on, but you should create statistics explicitly on columns used in filters and joins, and update them after large loads. Result-set caching keeps the result of a repeated query on the control node; identical queries are served without touching distributions until the underlying data changes. Workload management groups queries into workload groups with resource percentages, and classifiers route a session into a group by user, role, or label, so an ETL job cannot starve interactive BI.

**Worked example.** Create a fact table, a replicated dimension, statistics on a filtered column, and a workload group for the ETL login.

```sql
CREATE TABLE dbo.fact_sales
WITH (DISTRIBUTION = HASH(customer_id), CLUSTERED COLUMNSTORE INDEX)
AS SELECT * FROM staging.fact_sales;

CREATE TABLE dbo.dim_date
WITH (DISTRIBUTION = REPLICATE, CLUSTERED COLUMNSTORE INDEX)
AS SELECT * FROM staging.dim_date;

CREATE STATISTICS st_fact_sales_region ON dbo.fact_sales (region);
UPDATE STATISTICS dbo.fact_sales;

CREATE WORKLOAD GROUP wg_etl WITH (MIN_PERCENTAGE_RESOURCE = 20, CAP_PERCENTAGE_RESOURCE = 40, REQUEST_MIN_RESOURCE_GRANT_PERCENT = 5);
CREATE WORKLOAD CLASSIFIER cls_etl WITH (WORKLOAD_GROUP = 'wg_etl', MEMBERNAME = 'etl_login');
```

**How to read the plan and the DMVs.** Start with `EXPLAIN` or the estimated execution plan in the portal, and look for `ShuffleMove`, `BroadcastMove`, and `TrimMove` operations. A large move on the biggest table usually means the distribution key does not match the dominant join.

The DMVs tell you where time went and whether the data is even. `sys.dm_pdw_exec_requests` lists requests and their status, `sys.dm_pdw_request_steps` breaks a request into steps, `sys.dm_pdw_sql_requests` shows each distribution's share, `sys.dm_pdw_dms_workers` reports per-worker data movement, and `sys.dm_pdw_nodes_db_partition_stats` gives row counts per distribution for skew checks. Workload group and classifier state is in `sys.workload_management_workload_groups` and `sys.workload_management_workload_classifiers`.

```sql
SELECT request_id, status, total_elapsed_time, submit_time
FROM sys.dm_pdw_exec_requests
WHERE status NOT IN ('Completed', 'Failed', 'Cancelled')
ORDER BY submit_time DESC;

SELECT distribution_id, SUM(row_count) AS rows_per_distribution
FROM sys.dm_pdw_nodes_db_partition_stats
WHERE object_id = OBJECT_ID('dbo.fact_sales')
GROUP BY distribution_id
ORDER BY rows_per_distribution DESC;
```

**Common mistake.** Hash-distributing on a column people filter on but do not join on, or round-robin distributing every table and then joining large tables. Both force a shuffle on every query. A second mistake is trusting a stale statistic: after a bulk load the optimizer may still assume a small table and choose a broadcast that overflows memory.

**Gotcha.** A replicated table must stay small. Replication multiplies the table by 60, so a dimension that grows past a few gigabytes slows down loads and wastes space across the whole pool.

In [ ]:
REPLICATE_MAX_GB = 2.0

fact = {"name": "fact_sales", "rows": 800_000_000, "size_gb": 240.0, "join_key": "customer_id"}
dimensions = [
    {"name": "dim_date", "rows": 7_300, "size_gb": 0.02, "join_key": "date_key"},
    {"name": "dim_customer", "rows": 4_000_000, "size_gb": 4.0, "join_key": "customer_id"},
    {"name": "dim_product", "rows": 250_000, "size_gb": 0.3, "join_key": "product_key"},
]


def choose_distribution(table, fact):
    if table["size_gb"] <= REPLICATE_MAX_GB:
        return "REPLICATE"
    if table["join_key"] == fact["join_key"]:
        return f"HASH({table['join_key']})"
    return "ROUND_ROBIN"


print("fact:", f"HASH({fact['join_key']})")
for dim in dimensions:
    print(f"{dim['name']:<14} {choose_distribution(dim, fact)}")

assert choose_distribution(dimensions[0], fact) == "REPLICATE"
assert choose_distribution(dimensions[1], fact) == "HASH(customer_id)"
assert choose_distribution(dimensions[2], fact) == "REPLICATE"

## Migrating from Synapse to Fabric

**Mental model.** Migrate by workload, not by workspace. A Synapse workspace bundles four different things, and each has its own target in Fabric:

| Synapse component | Fabric target | Route |
|---|---|---|
| Dedicated SQL pool | Warehouse | Fabric Migration Assistant (schema from a DACPAC or a live connection, then a copy job for data) |
| Serverless SQL over ADLS | Lakehouse with shortcuts, queried through its SQL analytics endpoint | create shortcuts to the same ADLS paths; rewrite `OPENROWSET` views as endpoint views |
| Synapse Spark notebooks and jobs | Fabric notebooks and Spark job definitions | port code; replace Synapse-specific utilities and linked-service paths with OneLake paths |
| Synapse pipelines | Fabric Data Factory pipelines | follow Microsoft's pipeline upgrade guide; consider copy jobs or mirroring where they replace a pipeline outright |

**How the Migration Assistant works.** It runs in four steps: migrate object schemas into a new warehouse, fix the scripts it could not convert (with optional Copilot suggestions, prioritized so that fixing one primary object unblocks its dependents), copy data with a Fabric copy job, then test both warehouses side by side and reroute client connections. Expect manual work where Fabric differs: SQL-authenticated users become Entra ID users, column-level encryption needs another control such as dynamic data masking, `IDENTITY` columns behave differently, and external tables and multi-statement table-valued functions are not supported. Distribution clauses and indexes are simply dropped, because Fabric manages layout itself.

**Worked example.** A 6 TB dedicated pool feeds Power BI. Export a DACPAC, run the Migration Assistant into a new warehouse, fix the dozen stored procedures it flags, copy data with a copy job, and run the same report queries against both warehouses for two weeks, comparing row counts and totals per mart. Then repoint the semantic models and pause the dedicated pool before deleting it.

**Common mistake.** Treating the migration as a lift and shift of distributions and indexes. They have no meaning in a Fabric Warehouse; the work that matters is T-SQL compatibility, security mapping, and capacity sizing for the new workload.

**Gotcha.** The capacity bill replaces the DWU bill, but it is shared. A migrated warehouse that runs heavy loads at 08:00 competes with every report on the same capacity; put production BI and heavy engineering on separate capacities, or size for both.

## Tuning Azure Data Factory copy performance

**Mental model.** A copy activity is a metered reader and writer. Throughput is the product of how many parallel streams you allow and how fast each stream runs, not the size of the source. Data integration units (DIUs) budget compute for a cloud copy; copy parallelism controls how many concurrent connections read and write; and staging lets a warehouse load use its fastest bulk path.

**How it works.** For a cloud-to-cloud copy, raise `dataIntegrationUnits` and `parallelCopies` together until the source or sink saturates. If the source is on-premises, the bottleneck is usually the self-hosted integration runtime: it runs on your network, and a large copy needs several IR nodes behind a single logical runtime so the work spreads. For file stores, partition the copy so each run handles a slice, either with a `ForEach` over folders or with partition options.

Staging matters when the sink is a Synapse dedicated pool. `enableStaging: true` writes the data to Blob or ADLS first and then loads it with PolyBase or `COPY INTO`, which is much faster than row-by-row insert. Mapping data flows are a different engine: they run on a Spark cluster and bill by vCore-hour, so they should do real transform work rather than a simple copy.

**Worked example.** A copy of Parquet from an on-premises share to ADLS with staging, explicit DIUs, and parallelism.

```json
{
  "name": "CopyOrdersToLake",
  "type": "Copy",
  "typeProperties": {
    "source": {"type": "ParquetSource", "storeSettings": {"type": "FileServerReadSettings", "recursive": true}},
    "sink": {"type": "ParquetSink", "storeSettings": {"type": "AzureBlobFSWriteSettings"}},
    "dataIntegrationUnits": 32,
    "parallelCopies": 8,
    "enableStaging": true,
    "stagingSettings": {
      "linkedServiceName": {"referenceName": "LS_StagingLake", "type": "LinkedServiceReference"},
      "path": "staging/orders"
    }
  }
}
```

```bash
az datafactory pipeline create-run --resource-group rg-data --factory-name adf-acme   --name PL_Orders_Daily --parameters '{"windowStart":"2026-10-04"}'

az datafactory activity-run query-by-pipeline-run --resource-group rg-data --factory-name adf-acme   --run-id <run-id> --last-updated-after 2026-10-04T00:00:00Z --last-updated-before 2026-10-05T00:00:00Z
```

A tumbling-window trigger is the right choice when each run owns one time slice and you need exactly-once behavior with backfill; a schedule trigger is simpler when runs can overlap harmlessly. Blob-created triggers are event-driven and good for landing-zone arrival, but they can fire several times for one file, so make the downstream write idempotent.

**Common mistake.** Raising DIUs while `parallelCopies` stays at one, or leaving a single self-hosted IR node for a multi-terabyte copy. Both leave the bottleneck at the reader. Another mistake is enabling staging without checking that the staging path is network-reachable and that the workspace identity can write to it.

**Gotcha.** More DIUs do not fix a slow source. Measure the source read rate and the sink write rate separately; whichever saturates first is the one to change.

In [ ]:
def dius_for_copy(volume_gb, window_minutes, mbps_per_diu=6.0, max_dius=256):
    mb = volume_gb * 1024
    seconds = window_minutes * 60
    needed = mb / (seconds * mbps_per_diu)
    return min(max_dius, int(needed) + 1)


small = dius_for_copy(2048, 240)
large = dius_for_copy(1024, 30)
print(f"2 TB in 4 h   -> {small} DIUs")
print(f"1 TB in 30 min -> {large} DIUs")

assert small == 25
assert large == 98
assert dius_for_copy(2048, 240, max_dius=16) == 16

## Sizing Event Hubs throughput and checkpointing

**Mental model.** Event Hubs scales on two axes: capacity units, which set the total bandwidth of the namespace, and partitions, which set the maximum parallelism of consumers within a consumer group. Capacity units are throughput units (TU) on Standard, processing units (PU) on Premium, and capacity units (CU) on Dedicated.

**How it works.** One Standard throughput unit carries up to 1 MB/s or 1,000 events per second of ingress, whichever comes first, and up to 2 MB/s or 4,096 events per second of egress. A namespace can hold up to 40 TUs, and auto-inflate raises the count when ingress grows. Premium is isolated per namespace and a single processing unit offers roughly 5 to 10 MB/s of ingress and 10 to 20 MB/s of egress; Dedicated gives whole capacity units. Partition count does not change cost, but it caps consumer parallelism: one epoch consumer owns a partition at a time, so extra readers sit idle.

Checkpointing is what makes replay safe. The recommended consumer is the EventProcessorClient with a blob-backed checkpoint store: it takes leases on partitions and writes the last processed offset, so a restart resumes near where it stopped. Checkpoint too often and you pay extra storage operations; too rarely and a crash replays a large batch. Backpressure appears as `ServerBusy` on the producer side when ingress exceeds purchased capacity, and as rising consumer lag when processors cannot keep up. Capture is the escape hatch: it writes Avro micro-batches to ADLS or Blob Storage on a time and size window, so the raw stream is durable even if a consumer is down.

**Worked example.** Size a namespace for 18 MB/s and 12,000 events per second of ingress with a 30 MB/s egress read, then add headroom.

```bash
az eventhubs namespace create --resource-group rg-data --name eh-acme --sku Standard --capacity 20 --enable-auto-inflate true --maximum-throughput-units 40

az eventhubs eventhub create --resource-group rg-data --namespace-name eh-acme \
  --name orders --partition-count 32 --message-retention 7

az eventhubs eventhub consumer-group create --resource-group rg-data \
  --namespace-name eh-acme --eventhub-name orders --name spark
```

Ingress needs 18 TUs from bandwidth and 12 TUs from event rate, so 18 is the floor. Egress of 30 MB/s needs 15 TUs, below ingress. With 1.5x headroom the target is 27 TUs, and Standard allows up to 40. Choose at least 27 partitions so one consumer per partition can keep up, and remember that partition count is fixed on Standard.

```python
def throughput_units(ingress_mbps, ingress_events_per_s, egress_mbps, headroom=1.0):
    ingress = max(ingress_mbps / 1.0, ingress_events_per_s / 1000.0)
    egress = egress_mbps / 2.0
    return max(ingress, egress) * headroom

print(round(throughput_units(18, 12000, 30, 1.5)))
assert round(throughput_units(18, 12000, 30, 1.5)) == 27
```

**Common mistake.** Sizing on average load instead of peak, so producers throttle with `ServerBusy` during the daily spike. Another is deploying more consumer instances than partitions and expecting more throughput; the extra instances stay idle.

**Gotcha.** A checkpoint records a position in the stream, not a transaction. If a processor crashes after processing and before checkpointing, those events are delivered again. Make the sink idempotent, or deduplicate on an event ID.

In [ ]:
def throughput_units(ingress_mbps, ingress_events_per_s, egress_mbps, headroom=1.0):
    ingress = max(ingress_mbps / 1.0, ingress_events_per_s / 1000.0)
    egress = egress_mbps / 2.0
    return max(ingress, egress) * headroom


peak = throughput_units(18, 12_000, 30, headroom=1.5)
partitions = 32
standard_cap = 40
print(f"ingress-limited TUs with 1.5x headroom = {peak:.0f}")
print(f"partitions = {partitions}, Standard TU cap = {standard_cap}")

assert round(peak) == 27
assert peak <= standard_cap
assert partitions >= round(peak)

## Laying out ADLS Gen2 files for Synapse and Spark

**Mental model.** Engines pay per file, not per byte. Every file adds a listing, a metadata read, and a task; thousands of tiny files turn a fast query into a scheduling problem. The layout that performs is a small number of large files, partitioned only on columns that queries actually filter.

**How it works.** Partition directories such as `ds=2026-10-04/` let a query prune whole prefixes, but only when the filter matches the partition column. Partitioning on a high-cardinality column creates millions of tiny directories, which is worse than no partitioning. For file size, aim for a few hundred megabytes per file; 128 MB is a common floor and 512 MB to 1 GB suits large scans. Compaction rewrites many small files into fewer large ones, and Delta Lake does this with `OPTIMIZE`.

ADLS Gen2 supports lifecycle rules that move data between Hot, Cool, Cold, and Archive. Cool has a 30-day minimum, Cold has a 90-day minimum, and Archive has a 180-day minimum; deleting or moving a blob before the minimum triggers an early deletion charge. Archive data cannot be read until it is rehydrated, which can take hours, so it belongs on raw or compliance data, never on a table a query might need.

**Worked example.** Query a partitioned Parquet prefix in serverless SQL, and archive the raw zone on a schedule.

```sql
SELECT ds, COUNT(*) AS rows_in_partition
FROM OPENROWSET(
  BULK 'https://acmedata.dfs.core.windows.net/lake/curated/orders/ds=*/',
  FORMAT = 'PARQUET'
) WITH (order_id BIGINT, customer_id BIGINT, amount DECIMAL(12,2), ds VARCHAR(10) '$.ds')
AS orders
GROUP BY ds;
```

```json
{
  "rules": [
    {
      "name": "raw-orders-tiering",
      "enabled": true,
      "type": "Lifecycle",
      "definition": {
        "filters": {"blobTypes": ["blockBlob"], "prefixMatch": ["lake/raw/orders/"]},
        "actions": {
          "baseBlob": {
            "tierToCool": {"daysAfterModificationGreaterThan": 30},
            "tierToArchive": {"daysAfterModificationGreaterThan": 180},
            "delete": {"daysAfterModificationGreaterThan": 365}
          }
        }
      }
    }
  ]
}
```

The curated zone stays Hot because queries read it daily, the raw zone moves to Cool and then Archive, and the deletion rule enforces retention. Serverless SQL bills per terabyte scanned, so a narrow projection and a matching partition filter are what keep the query cheap.

**Common mistake.** Landing every micro-batch as its own file, then querying the folder with serverless SQL. The engine spends more time opening files than reading them. A second mistake is partitioning a small table by a high-cardinality key such as an order ID.

**Gotcha.** Lifecycle policies are account-wide and irreversible for a blob that is already archived. Test a new rule against a small prefix first, and remember that a query against archived data fails until the blob is rehydrated.

In [ ]:
def plan_compaction(total_mb, file_count, target_mb=256):
    current_mb = total_mb / file_count
    target_files = max(1, -(-int(total_mb) // target_mb))
    return current_mb, target_files


total = 120_000 * 4
current, target = plan_compaction(total, 120_000)
print(f"current file size = {current:.0f} MB across 120,000 files")
print(f"after compaction   = {target} files of about {total / target:.0f} MB")
print(f"file-count reduction = {120_000 / target:.0f}x")

assert current == 4
assert target == 1875
assert round(120_000 / target) == 64

## Things to remember: key numbers and limits

Numbers that come up in design reviews. Prices are illustrative list prices at the time of writing; limits are stable.

| Topic | Number / fact | Why it matters |
|---|---|---|
| Fabric capacity | F SKUs from F2 (2 CUs), doubling per step; billed per second, 1-minute minimum; pausable | one pool pays for every engine in its workspaces |
| Fabric smoothing | interactive over 5-64 minutes, background over 24 hours; 30-second timepoints | size from daily background CU-seconds, not the first-hour peak |
| Fabric throttling | delay after 10 minutes of future capacity used, interactive rejection after 60, all rejected after 24 hours | running jobs finish; new ones wait or fail |
| Fabric viewers | free-license viewers can read Power BI content on F64 and larger | below F64 every viewer needs Pro or PPU |
| Queue Storage | messages up to 64 KB, at-least-once delivery, visibility timeout | a decoupling queue, not an analytics stream; use Event Hubs for replay |
| Event Hubs TU | 1 MB/s ingress and 2 MB/s egress per throughput unit; Standard caps at 40 TUs | size on peak, not average; partitions cap per-consumer-group parallelism |
| Event Hubs partitions | fixed at creation unless auto-inflate; ordering is per partition key | choose partition count for peak consumer parallelism |
| Synapse dedicated pool | every table spread across 60 distributions; replicate dimensions under ~2 GB | distribution choice decides whether big joins shuffle |
| Synapse concurrency | DW1000c allows 32 concurrent queries; scale up for more | concurrency, not raw speed, is often the sizing driver |
| ADF copy | about 6 MB/s per DIU, cap 256 DIUs; use staging for on-prem or large copies | compute DIUs from volume and window |
| ADLS tiers | Cool needs 30 days untouched, Cold 90, Archive 180; Archive rehydrate takes hours | tiering short-lived raw data wastes money on early-deletion fees |
| ADLS security | effective permission = union of RBAC and POSIX ACLs | an RBAC role can silently widen an ACL lockdown |
| File size target | roughly 128 MB to 1 GB Parquet files | engines pay per file in listing and planning, not per byte |

## Things to remember: semantics to memorize

* **Fabric is the default for new analytics; Synapse is the existing path.** Run inherited Synapse pools well, but design new warehouses on Fabric and migrate by workload.
* **A lakehouse SQL analytics endpoint is read-only.** Write with Spark, pipelines or dataflows; use a Warehouse when you need T-SQL DML and multi-table transactions.
* **Shortcuts are links, not copies.** Deleting a shortcut leaves the target alone; deleting files *inside* a shortcut deletes them at the target if you have permission.
* **Event Hubs is a replayable log, not a queue.** Queue Storage deletes on dequeue; Event Hubs retains for the retention period so consumers can rewind.
* **Managed identity over keys everywhere.** `--auth-mode login`, Key Vault for secrets, RBAC at the tightest scope; account keys in pipeline configs are the recurring incident.
* **RBAC and ACLs are additive on ADLS.** The effective permission is the union — tightening ACLs does not undo a broad RBAC grant.
* **Synapse serverless bills per TB processed**; a dedicated pool bills per hour while online — pause it or pay for idle weekends.
* **Dead-letter queues need a monitor and a replay path.** A growing DLQ is a silent outage; the messages are parked, not lost.
* **Distribution follows the dominant join** in a Synapse dedicated pool: hash on the join key, replicate small dimensions, round-robin only for staging.

## Real-world use cases on Azure

### Use case 1: A Fabric lakehouse over an existing ADLS lake

Business context: a distributor already lands ERP and e-commerce data in ADLS Gen2 and wants self-service SQL and Power BI without another copy. Data volume: ~15 TB, 250 tables.

Architecture: a Fabric Lakehouse holds shortcuts to the curated Delta folders in ADLS; Spark notebooks build silver tables; a Fabric Warehouse holds the gold star schema built with cross-database queries; Power BI semantic models read it in Direct Lake mode; a copy job brings in two SaaS sources the lake did not have.

```sql
-- Fabric Warehouse: build a gold table from the lakehouse without copying the source files.
CREATE TABLE gold.fact_sales AS
SELECT s.order_id, s.customer_id, CAST(s.order_ts AS DATE) AS order_date, s.amount
FROM erp_lakehouse.dbo.sales_silver AS s;
```

Why: shortcuts reuse the existing lake, the Warehouse gives analysts full T-SQL and transactions for the served model, and one capacity covers engineering, SQL and reporting.

### Use case 2: Nightly retail ELT into an existing Synapse warehouse

Business context: a retailer consolidates point-of-sale, e-commerce, and inventory into one reporting layer for merchandising. Data volume: ~2 TB/night, ~40 source systems.

Architecture: Data Factory copies sources to `raw/` in ADLS Gen2, Databricks cleans and models to Delta in `curated/`, and Synapse SQL dedicated pools serve Power BI through `COPY INTO`.

```sql
COPY INTO dbo.fact_sales
FROM 'https://acmedata.dfs.core.windows.net/lake/curated/fact_sales/'
WITH (FILE_TYPE = 'PARQUET', CREDENTIAL = (IDENTITY = 'Managed Identity'));
```

Why: ADF handles breadth of connectors cheaply, Databricks absorbs the heavy transformation, and Synapse SQL gives sub-second dashboard queries with hash-distributed facts and replicated dimensions. This is the estate the Migration Assistant targets: the dedicated pool becomes a Fabric Warehouse while ADLS and Databricks stay.

### Use case 3: IoT telemetry streaming with Event Hubs and Spark

Business context: a manufacturer monitors equipment health and predicts failures. Data volume: ~5 million events/minute, ~30 TB/month.

Architecture: devices -> IoT Hub/Event Hubs (8 or more partitions) -> Databricks Structured Streaming with a Delta sink; Event Hubs Capture archives raw Avro to ADLS for replay; Synapse or Power BI reads aggregates.

```python
events = spark.readStream.format("eventhubs").option("eventhubs.connectionString", connection_string).load()
(events.writeStream.format("delta")
 .option("checkpointLocation", "abfss://lake@acmedata.dfs.core.windows.net/checkpoints/telemetry")
 .trigger(availableNow=True)
 .toTable("curated.telemetry"))
```

Why: Event Hubs absorbs bursty device traffic with ordering per partition key, Structured Streaming gives exactly-once Delta writes with checkpointing, and Capture keeps the raw stream available for reprocessing after a model change.

### Use case 4: Governed self-service analytics with Purview and Unity Catalog

Business context: a bank wants analysts to discover data without exposing PII, under regulatory audit. Data volume: ~50 TB across 300 tables.

Architecture: ADLS Gen2 registered in Unity Catalog, Purview scanning and classifying PII, Entra ID groups mapped to Unity Catalog grants, and Power BI consuming curated marts.

```bash
az role assignment create --assignee <analyst-group-id> --assignee-principal-type Group \
  --role "Storage Blob Data Reader" \
  --scope /subscriptions/<sub-id>/resourceGroups/rg-data/providers/Microsoft.Storage/storageAccounts/acmedata
```

Why: Unity Catalog centralizes table and column grants and lineage, Purview provides classification and data mapping for compliance, and Entra ID groups keep access management aligned with HR-driven identity.

### Use case 5: Cost-controlled log analytics with serverless SQL

Business context: an SRE team needs 12 months of application logs for incident response without a standing warehouse. Data volume: ~30 TB, bursty query patterns.

Architecture: diagnostic settings land logs in ADLS Gen2 as Parquet, Synapse SQL serverless queries them in place, and Workload Groups plus a per-query cost ceiling cap spend.

```sql
SELECT level, count(*) AS events
FROM OPENROWSET(BULK 'https://acmedata.dfs.core.windows.net/lake/logs/app/*/*.parquet', FORMAT = 'PARQUET') AS logs
WHERE status >= 500
GROUP BY level;
```

Why: serverless SQL avoids a provisioned warehouse for intermittent queries, Parquet plus directory partitioning keeps scans small, and lifecycle policies move older logs to cooler tiers.

## Azure exercises

These use only the standard library to model sizing and layout decisions. Prices are illustrative; treat each assert as a check on the formula.

### Try it 1. Choose ADLS access tiers over 12 months

**Task.** Assign each dataset to Hot, Cool, Cold, or Archive from its days since last read, then compare 12 months of storage cost with keeping everything Hot. Use illustrative prices per GB per month: Hot 0.02, Cool 0.01, Cold 0.005, Archive 0.002.

**Hint.** Archive needs at least 180 days since last read, Cold at least 90, and Cool at least 30. Anything read more recently stays Hot. Remember that moving a blob out of a tier before its minimum days incurs an early deletion charge.

**Expected result.** `raw_events` archives, `curated_sales` goes Cold, `gold_marts` goes Cool, and `live_landing` stays Hot. The tiered monthly cost is about 690 against 4,940 for all-Hot, a saving of about 4,250 per month.

In [ ]:
PRICE_PER_GB_MONTH = {"hot": 0.02, "cool": 0.01, "cold": 0.005, "archive": 0.002}
MIN_DAYS = {"hot": 0, "cool": 30, "cold": 90, "archive": 180}


def choose_tier(days_since_read):
    for tier in ("archive", "cold", "cool", "hot"):
        if days_since_read >= MIN_DAYS[tier]:
            return tier
    return "hot"


datasets = [
    ("raw_events", 200_000, 210),
    ("curated_sales", 40_000, 95),
    ("gold_marts", 5_000, 45),
    ("live_landing", 2_000, 0),
]

tiered = 0.0
all_hot = 0.0
for name, size_gb, days in datasets:
    tier = choose_tier(days)
    tiered += size_gb * PRICE_PER_GB_MONTH[tier]
    all_hot += size_gb * PRICE_PER_GB_MONTH["hot"]
    print(f"{name:<14} last read {days:>3}d -> {tier}")

saving = all_hot - tiered
print(f"tiered = {tiered:,.0f} vs all-hot = {all_hot:,.0f} per month")
print(f"saving = {saving:,.0f} per month")

assert choose_tier(210) == "archive"
assert choose_tier(95) == "cold"
assert choose_tier(45) == "cool"
assert choose_tier(0) == "hot"
assert round(tiered) == 690
assert round(saving) == 4250

### Try it 2. Detect skew across 60 distributions

**Task.** A fact table has one row count per distribution. Compute the mean, the ratio of the largest distribution to the mean, and flag skew when the ratio exceeds 1.5.

**Hint.** Build the list with 59 even distributions and one hot distribution, then use `max`, `sum`, and `len`.

**Expected result.** The mean is about 1,016,667 rows, the hot distribution has 2,000,000 rows, and the ratio is about 1.967, so the table is skewed.

In [ ]:
rows_per_distribution = [1_000_000] * 59 + [2_000_000]
DISTRIBUTION_COUNT = 60
SKEW_THRESHOLD = 1.5

total_rows = sum(rows_per_distribution)
mean_rows = total_rows / len(rows_per_distribution)
max_rows = max(rows_per_distribution)
ratio = max_rows / mean_rows
skewed = ratio > SKEW_THRESHOLD
hottest = rows_per_distribution.index(max_rows)

print(f"distributions = {len(rows_per_distribution)}")
print(f"mean = {mean_rows:,.0f} rows, max = {max_rows:,} rows on distribution {hottest}")
print(f"max/mean = {ratio:.3f}, skewed = {skewed}")

assert len(rows_per_distribution) == DISTRIBUTION_COUNT
assert round(ratio, 3) == 1.967
assert skewed is True
assert hottest == 59

### Try it 3. Size a Synapse DWU from concurrency

**Task.** Given the maximum concurrent queries documented per performance level, pick the smallest level that supports a peak of 30 concurrent queries, and the smallest that supports 90.

**Hint.** Sort the levels by concurrency and take the first one whose maximum is at least the peak. Remember that scaling up raises DWUs and lowers the per-query share for small levels, so concurrency does not grow linearly forever.

**Expected result.** 30 concurrent queries need DW1000c (32 maximum). 90 concurrent queries need DW6000c (128 maximum), because DW3000c tops out at 64.

In [ ]:
MAX_CONCURRENT_QUERIES = {
    "DW100c": 4,
    "DW200c": 8,
    "DW300c": 12,
    "DW400c": 16,
    "DW500c": 20,
    "DW1000c": 32,
    "DW2000c": 48,
    "DW3000c": 64,
    "DW6000c": 128,
}


def smallest_level(peak_concurrency):
    for level, limit in sorted(MAX_CONCURRENT_QUERIES.items(), key=lambda item: (item[1], item[0])):
        if limit >= peak_concurrency:
            return level, limit
    raise ValueError("beyond the largest level")


level_30, limit_30 = smallest_level(30)
level_90, limit_90 = smallest_level(90)
print(f"peak 30 -> {level_30} (max {limit_30})")
print(f"peak 90 -> {level_90} (max {limit_90})")

assert level_30 == "DW1000c"
assert level_90 == "DW6000c"
assert MAX_CONCURRENT_QUERIES["DW3000c"] < 90

### Try it 4. Cost a dedicated pool with pause

**Task.** A dedicated SQL pool costs an illustrative 6.00 per hour at DW1000c. Compare always-on (720 hours a month) with pausing outside a 10-hour business window on 22 working days. Report the monthly cost and the saving.

**Hint.** Running hours are 10 times 22. The pool bills by the hour while it is online, and there is no charge while it is paused, though a paused pool must be resumed before the next query.

**Expected result.** Always-on costs 4,320; business-hours-only costs 1,320, a saving of 3,000, or about 69 percent.

In [ ]:
RATE_PER_HOUR = 6.00
HOURS_PER_MONTH = 720
BUSINESS_HOURS = 10 * 22

always_on = HOURS_PER_MONTH * RATE_PER_HOUR
paused = BUSINESS_HOURS * RATE_PER_HOUR
saving = always_on - paused
saving_pct = saving / always_on * 100

print(f"always-on   = {always_on:,.0f} per month")
print(f"paused      = {paused:,.0f} per month ({BUSINESS_HOURS} hours)")
print(f"saving      = {saving:,.0f} per month ({saving_pct:.1f}%)")

assert always_on == 4320
assert paused == 1320
assert saving == 3000
assert round(saving_pct, 1) == 69.4

### Try it 5. Pick a Fabric capacity from daily background usage

**Task.** A team's nightly pipelines, Spark jobs and warehouse loads consume 4,200,000 CU-seconds of background work per day. Background work is smoothed over 24 hours, so its average utilization is the daily CU-seconds divided by the CU-seconds the capacity provides in a day. Pick the smallest F SKU that keeps that average at or below 80%, leaving headroom for interactive reports.

**Hint.** An F SKU with `n` CUs provides `n * 86,400` CU-seconds per day. Try F16, F32, F64 and F128 in order.

**Expected result.** F32 would run at about 152% (it would throttle), and F64 runs at about 76%, so F64 is the smallest SKU that fits.

In [ ]:
SECONDS_PER_DAY = 86_400
TARGET_UTILIZATION = 0.80
DAILY_BACKGROUND_CU_SECONDS = 4_200_000
SKUS = {"F16": 16, "F32": 32, "F64": 64, "F128": 128}


def utilization(cu_seconds_per_day, capacity_cus):
    return cu_seconds_per_day / (capacity_cus * SECONDS_PER_DAY)


def smallest_sku(cu_seconds_per_day):
    for sku, cus in sorted(SKUS.items(), key=lambda item: item[1]):
        if utilization(cu_seconds_per_day, cus) <= TARGET_UTILIZATION:
            return sku
    raise ValueError("needs a larger capacity than F128")


for sku, cus in SKUS.items():
    print(f"{sku:<5} {utilization(DAILY_BACKGROUND_CU_SECONDS, cus):6.1%}")
choice = smallest_sku(DAILY_BACKGROUND_CU_SECONDS)
print("smallest SKU at or below 80%:", choice)

assert round(utilization(DAILY_BACKGROUND_CU_SECONDS, 32), 2) == 1.52
assert round(utilization(DAILY_BACKGROUND_CU_SECONDS, 64), 2) == 0.76
assert choice == "F64"